# Featuresmith Tutorial: 01 — Getting Started with Featuresmith v0.4.0

Learn the fundamentals of Featuresmith — loading tabular datasets, inspecting Dataset descriptors, deterministic statistical profiling, automated dataset code reviews, ML readiness scoring, recommendations, and the Plan primitive.

---


## 1. Problem Statement & Why This Capability Matters
Machine learning failure modes often originate from dataset quality rather than model architecture choices. Featuresmith brings developer-first code review discipline to tabular datasets before training begins.

### Objectives
1. Understand Featuresmith's package structure (`featuresmith-core` and `featuresmith-cli`).
2. Load tabular data from CSV, Parquet, Excel, or DataFrames using `fs.load()`.
3. Inspect `Dataset` properties (`row_count`, `column_count`, `dtypes`, `source`, `preview()`).
4. Profile datasets deterministically with `fs.profile()`.
5. Perform an automated dataset code review with `fs.review()`.
6. Extract an explainable 0–100 ML Readiness Score with `fs.score()`.

### Step 1: Import Featuresmith & Verify Version

In [1]:
import os

import pandas as pd

import featuresmith as fs

print(f"Featuresmith Version: {fs.__version__}")

Featuresmith Version: 0.4.0


### Step 2: Load Tabular Datasets & Inspect Dataset Objects
`fs.load()` normalizes local files (`.csv`, `.parquet`, `.xlsx`) and in-memory DataFrames into a shallowly immutable `Dataset` contract without copying memory buffers.

In [2]:
data_path = os.path.join("..", "data", "processed", "titanic.csv")
dataset = fs.load(data_path)

print(f"Dataset Source : {dataset.source}")
print(f"Backend Engine : {dataset.backend}")
print(f"Row Count      : {dataset.row_count}")
print(f"Column Count   : {dataset.column_count}")
print(f"Columns        : {dataset.schema.names}")

# Preview first 3 rows
print("\nData Preview:")
print(dataset.preview(3))

Dataset Source : ..\data\processed\titanic.csv
Backend Engine : polars
Row Count      : 891
Column Count   : 12
Columns        : ('passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked')

Data Preview:
shape: (3, 12)
┌─────────────┬──────────┬────────┬───────────────────┬───┬───────────┬─────────┬───────┬──────────┐
│ passengerid ┆ survived ┆ pclass ┆ name              ┆ … ┆ ticket    ┆ fare    ┆ cabin ┆ embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---               ┆   ┆ ---       ┆ ---     ┆ ---   ┆ ---      │
│ i64         ┆ i64      ┆ i64    ┆ str               ┆   ┆ str       ┆ f64     ┆ str   ┆ str      │
╞═════════════╪══════════╪════════╪═══════════════════╪═══╪═══════════╪═════════╪═══════╪══════════╡
│ 1           ┆ 0        ┆ 3      ┆ Braund, Mr. Owen  ┆ … ┆ A/5 21171 ┆ 7.25    ┆ null  ┆ S        │
│             ┆          ┆        ┆ Harris            ┆   ┆           ┆         ┆       ┆          │
│ 2           ┆ 1

### Step 3: Load In-Memory DataFrame using `from_dataframe` or `load`

In [3]:
df = pd.DataFrame({"age": [25, 30, 35], "income": [50000.0, 65000.0, 80000.0]})
ds_mem = fs.load(df)
print(
    f"In-Memory Dataset Row Count: {ds_mem.row_count}, Columns: {ds_mem.column_count}"
)

In-Memory Dataset Row Count: 3, Columns: 2


### Step 4: Run Vectorized Deterministic Profiling
`fs.profile()` computes statistical descriptors (min, max, mean, quantiles, missingness, cardinality, correlations) deterministically.

In [4]:
profile = fs.profile(dataset)
print(f"Overall Missingness: {profile.dataset_summary.missing_percentage:.2f}%")
print("\nSample Column Profiles:")
for col, col_prof in list(profile.column_profiles.items())[:5]:
    print(
        f"  - {col:<15}: logical_type={col_prof.logical_type:<12} missing={col_prof.missing_count}"
    )

Overall Missingness: 8.10%

Sample Column Profiles:
  - passengerid    : logical_type=numeric      missing=0
  - survived       : logical_type=numeric      missing=0
  - pclass         : logical_type=numeric      missing=0
  - name           : logical_type=text         missing=0
  - sex            : logical_type=categorical  missing=0


### Step 5: Run Automated Dataset Review & Scorecard
`fs.review()` evaluates dataset health across 10 specialized reviewers (including Feature Quality in v0.4.0), while `fs.score()` extracts an overall ML Readiness Scorecard. (Passing `previous=` to `fs.review()` also activates the DiffReviewer in v0.3.0+.)

In [5]:
review_result = fs.review(dataset, target_column="survived")
scorecard = fs.score(review_result)

if scorecard:
    print(f"ML Readiness Score: {scorecard.overall:.1f} / 100")
    print("\nDimension Breakdown:")
    for dim in scorecard.dimensions:
        print(
            f"  - {dim.label:<20}: {dim.score:5.1f}/100 ({len(dim.contributing_findings)} findings)"
        )

ML Readiness Score: 85.0 / 100

Dimension Breakdown:
  - Schema Health       : 100.0/100 (0 findings)
  - Missing Values      :  70.0/100 (1 findings)
  - Feature Quality     : 100.0/100 (0 findings)
  - Distribution Health :  45.0/100 (5 findings)
  - Leakage Risk        : 100.0/100 (0 findings)
  - Data Quality        : 100.0/100 (0 findings)
  - Consistency         :  80.0/100 (4 findings)


In [6]:
review_result = fs.review(dataset, target_column="survived")

# Inspect recommendations
print("Recommendations from Review:")
for rec in review_result.recommendations:
    print(f"  [{rec.severity.upper()}] {rec.title} — {rec.suggested_action}")

# Create a Plan from accepted recommendations
if review_result.recommendations:
    plan = fs.plan(review_result, accept=[review_result.recommendations[0].id])
    print(f"\nPlan created with {len(plan.items)} item(s):")
    for item in plan.items:
        print(f"  {item.id}: {item.title}")

Recommendations from Review:
  [CRITICAL] Fix missing value threshold in column 'cabin' — Address the flagged issue: High missing values in column 'cabin'.
  [WARNING] Fix basic statistics in column 'sibsp' — Address the flagged issue: High skewness in column 'sibsp'.
  [WARNING] Fix basic statistics in column 'parch' — Address the flagged issue: High skewness in column 'parch'.
  [WARNING] Fix basic statistics in column 'fare' — Address the flagged issue: High skewness in column 'fare'.
  [INFO] Fix data types in column 'passengerid' — Address the flagged issue: Identifier-like column 'passengerid'.
  [INFO] Fix data types in column 'name' — Address the flagged issue: Text column 'name'.
  [INFO] Fix data types in column 'ticket' — Address the flagged issue: Text column 'ticket'.
  [INFO] Fix data types in column 'cabin' — Address the flagged issue: Text column 'cabin'.

Plan created with 1 item(s):
  plan.rec.quality.cabin.0: Fix missing value threshold in column 'cabin'


### Key Takeaways & Connection to Next Tutorial
- `fs.load()` normalizes files and DataFrames into a standard schema contract.
- `fs.profile()` executes ultra-fast computations to extract shape and column descriptors.
- `fs.review()` runs 10 automated reviewers (including Feature Quality in v0.4.0) to inspect missingness, data types, target leakage risk, and feature quality.
- `fs.score()` transforms review findings into an explainable 0–100 quality scorecard (7 dimensions in v0.4.0).
- `fs.plan()` compiles a deterministic, inspectable Plan from accepted recommendations with full traceability.

**Next Tutorial**: In `02_dataset_review.ipynb`, we explore the Review Engine's 10 automated reviewers, category filtering, reviewer configuration, recommendations, and text output rendering.